# Homework 5 - Model Deployment with MLflow & F1 Dataset

**Objectives**
1. Create two prediction tables in a personal database
2. Build two predictive models with MLflow (hyperparams, model, 4 metrics, 2 artifacts)
3. Write predictions into the corresponding tables
4. Push to GitHub


## 0  Setup - Install dependencies & imports

In [0]:
import os, tempfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             r2_score, median_absolute_error)

print('All imports OK')


## 1  Load the F1 Dataset

We use the Ergast F1 dataset stored in the course Volume.  
**Target variable**: `milliseconds` (race finishing time).  
Features: grid position, laps, year, round, circuit, driver & constructor IDs, nationalities, altitude.


In [0]:
results = spark.read.csv('/Volumes/gr5069/raw/f1_data/results.csv', 
                          header=True, inferSchema=True, nullValue='\\N')
races        = spark.read.csv('/Volumes/gr5069/raw/f1_data/races.csv',          header=True, inferSchema=True)
drivers      = spark.read.csv('/Volumes/gr5069/raw/f1_data/drivers.csv',        header=True, inferSchema=True)
constructors = spark.read.csv('/Volumes/gr5069/raw/f1_data/constructors.csv',   header=True, inferSchema=True)
circuits     = spark.read.csv('/Volumes/gr5069/raw/f1_data/circuits.csv',       header=True, inferSchema=True)

print('results :', results.count())
print('races   :', races.count())


In [0]:
from pyspark.sql.functions import col

df_spark = (
    results
    .join(races.select('raceId','year','circuitId','round'), on='raceId', how='left')
    .join(drivers.select('driverId','nationality'), on='driverId', how='left')
    .join(
        constructors.select('constructorId','nationality')
                    .withColumnRenamed('nationality','constructor_nationality'),
        on='constructorId', how='left'
    )
    .join(circuits.select('circuitId','country','alt'), on='circuitId', how='left')
    .select(
        'raceId','driverId','constructorId','circuitId',
        'grid','positionOrder','laps','milliseconds',
        'year','round',
        col('nationality').alias('driver_nationality'),
        'constructor_nationality',
        col('country').alias('circuit_country'),
        col('alt').alias('circuit_altitude')
    )
    .filter(col('milliseconds').isNotNull() & (col('milliseconds') > 0))
    .filter(col('grid').isNotNull()         & (col('grid') > 0))
)

df = df_spark.toPandas()
print(df.shape)
df.head(3)


## 2  Feature Engineering

In [0]:
cat_cols = ['driver_nationality', 'constructor_nationality', 'circuit_country']
le = LabelEncoder()
for c in cat_cols:
    df[c] = df[c].fillna('Unknown')
    df[c] = le.fit_transform(df[c].astype(str))

df['circuit_altitude'] = pd.to_numeric(df['circuit_altitude'], errors='coerce').fillna(0)

feature_cols = ['grid','laps','year','round','circuitId',
                'driverId','constructorId',
                'driver_nationality','constructor_nationality',
                'circuit_altitude']

X = df[feature_cols]
y = df['milliseconds']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')


## 3  Create Prediction Tables in Personal Database  *(20 pts)*

We create **two** Delta tables - one per model - in the student's own database.


In [0]:
MY_NETID = "zx2556"
BASE_PATH = f"/Volumes/gr5069/{MY_NETID}/takehome"
print(f"Data will be saved to: {BASE_PATH}")

## 4  MLflow Experiments  *(30 pts)*

Each run logs:
- **Hyperparameters** - model-specific params
- **4 Metrics** - MSE, RMSE, MAE, R2
- **2 Artifacts** - feature-importance CSV, residual plot PNG
- **Model** - `mlflow.sklearn.log_model`


In [0]:
def mlflow_log(experiment_id, run_name, model, params,
               X_train, X_test, y_train, y_test, feature_names):
    with mlflow.start_run(experiment_id=experiment_id, run_name=run_name) as run:

        # Train
        model.set_params(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        # Log model
        mlflow.sklearn.log_model(model, 'model')

        # Log hyperparameters
        for k, v in params.items():
            mlflow.log_param(k, v)

        # Log 4 metrics
        mse  = mean_squared_error(y_test, preds)
        rmse = float(np.sqrt(mse))
        mae  = mean_absolute_error(y_test, preds)
        r2   = r2_score(y_test, preds)
        mlflow.log_metric('mse',  mse)
        mlflow.log_metric('rmse', rmse)
        mlflow.log_metric('mae',  mae)
        mlflow.log_metric('r2',   r2)
        print(f'[{run_name}]  RMSE={rmse:.1f}  MAE={mae:.1f}  R2={r2:.4f}')

        # Artifact 1: feature importance CSV
        if hasattr(model, 'feature_importances_'):
            importance = (
                pd.DataFrame({'Feature': feature_names,
                              'Importance': model.feature_importances_})
                .sort_values('Importance', ascending=False)
            )
            tmp = tempfile.NamedTemporaryFile(suffix='.csv', delete=False)
            importance.to_csv(tmp.name, index=False)
            mlflow.log_artifact(tmp.name, 'feature_importance.csv')
            os.unlink(tmp.name)

        # Artifact 2: residual plot PNG
        fig, ax = plt.subplots(figsize=(7, 4))
        residuals = y_test.values - preds
        sns.residplot(x=preds, y=residuals, lowess=True, ax=ax,
                      line_kws={'color': 'red', 'lw': 1.5})
        ax.set_xlabel('Predicted milliseconds')
        ax.set_ylabel('Residual')
        ax.set_title(f'Residual Plot - {run_name}')
        plt.tight_layout()
        tmp2 = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
        fig.savefig(tmp2.name, dpi=100)
        mlflow.log_artifact(tmp2.name, 'residual_plot.png')
        os.unlink(tmp2.name)
        plt.show()

        return run.info.run_id, preds


In [0]:
# TODO: update to your Databricks email / username
EXPERIMENT_NAME = '/Users/zx2556@columbia.edu/hw5_f1_model_deployment'

mlflow.set_experiment(EXPERIMENT_NAME)
experiment    = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
experiment_id = experiment.experiment_id
print('Experiment ID:', experiment_id)


### Model 1 - Random Forest Regressor

In [0]:
rf_params = {
    'n_estimators': 300,
    'max_depth':    10,
    'random_state': 42,
    'n_jobs':       -1,
}

rf_model = RandomForestRegressor()
rf_run_id, rf_preds = mlflow_log(
    experiment_id, 'RF - F1 Lap Time',
    rf_model, rf_params,
    X_train, X_test, y_train, y_test, feature_cols
)
print('RF run_id:', rf_run_id)


### Model 2 - Gradient Boosting Regressor

In [0]:
gb_params = {
    'n_estimators':  300,
    'max_depth':       5,
    'learning_rate':   0.05,
    'subsample':       0.8,
    'random_state':   42,
}

gb_model = GradientBoostingRegressor()
gb_run_id, gb_preds = mlflow_log(
    experiment_id, 'GB - F1 Lap Time',
    gb_model, gb_params,
    X_train, X_test, y_train, y_test, feature_cols
)
print('GB run_id:', gb_run_id)


## 5  Write Predictions to Database Tables  *(30 pts)*

In [0]:
# 生成预测 DataFrame
test_meta = df.loc[X_test.index, ['raceId','driverId','constructorId','grid']].reset_index(drop=True)

def build_pred_df(meta, actuals, predictions, run_id):
    out = meta.copy()
    out['actual_milliseconds']    = actuals.values
    out['predicted_milliseconds'] = predictions
    out['residual']               = out['actual_milliseconds'] - out['predicted_milliseconds']
    out['run_id']                 = run_id
    out['created_at']             = pd.Timestamp.now()
    return spark.createDataFrame(out)

rf_pred_df = build_pred_df(test_meta, y_test.reset_index(drop=True), rf_preds, rf_run_id)
gb_pred_df = build_pred_df(test_meta, y_test.reset_index(drop=True), gb_preds, gb_run_id)

# 写入 Volume
rf_pred_df.write.format("delta").mode("overwrite").save(f"{BASE_PATH}/rf_predictions")
gb_pred_df.write.format("delta").mode("overwrite").save(f"{BASE_PATH}/gb_predictions")

print(f"RF predictions saved to: {BASE_PATH}/rf_predictions")
print(f"GB predictions saved to: {BASE_PATH}/gb_predictions")

In [0]:
%sql
SHOW VOLUMES IN gr5069.zx2556

In [0]:
%sql
CREATE VOLUME gr5069.zx2556.takehome

## 6  Verify Tables

In [0]:
# Verify Tables
rf_check = spark.read.format("delta").load(f"{BASE_PATH}/rf_predictions")
gb_check = spark.read.format("delta").load(f"{BASE_PATH}/gb_predictions")

print(f"RF predictions count: {rf_check.count()}")
display(rf_check.limit(5))

print(f"GB predictions count: {gb_check.count()}")
display(gb_check.limit(5))